In [ ]:
SELECT CURRENT_ROLE()

In [ ]:
DROP DATABASE IF EXISTS DEMO_DB;

## Step 1: Setup

In [ ]:
-- Create new database and schema

CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

## Step 2: Create Source Table and Materialized View

In [ ]:
-- Create a source table to store sales data:

CREATE TABLE sales (
    sale_id INT,
    product_name STRING,
    amount DECIMAL(10,2),
    sale_date DATE
);

In [ ]:
-- Create a materialized view that aggregates sales by product

CREATE MATERIALIZED VIEW sales_by_product
    AS
    SELECT product_name, SUM(amount) AS total_sales
    FROM sales
    GROUP BY product_name;

## Step 3: Insert and Query Data

In [ ]:
INSERT INTO sales VALUES
    (1, 'Laptop', 1000.00, '2025-09-20'),
    (2, 'Laptop', 1500.00, '2025-09-20'),
    (3, 'Phone', 800.00, '2025-09-21'),
    (4, 'Phone', 900.00, '2025-09-21');

In [ ]:
SELECT * FROM sales_by_product;

/*

- Materialized views automatically refresh when the source table changes, managed by Snowflake’s background services (no manual refresh needed, unlike dynamic tables).
- Refreshes are incremental, making materialized views efficient for aggregations over large datasets.
- Querying the materialized view is faster than querying the source table with the same aggregation because the results are precomputed.

*/


## Step 4: Explore Materialized View Behavior

In [ ]:
SELECT * 
FROM TABLE(INFORMATION_SCHEMA.MATERIALIZED_VIEW_REFRESH_HISTORY())
WHERE MATERIALIZED_VIEW_NAME = 'SALES_BY_PRODUCT';

In [ ]:
-- Test Source Table Modifications:
UPDATE sales SET amount = 1200.00 WHERE sale_id = 1;
DELETE FROM sales WHERE sale_id = 2;

In [ ]:
-- Wait briefly (e.g., 10–30 seconds) for Snowflake to refresh the materialized view, then query it:
SELECT * FROM sales_by_product;

In [ ]:
-- Test Limitations:

CREATE OR REPLACE MATERIALIZED VIEW invalid_mv
    AS
    SELECT product_name, SUM(amount) AS total_sales
    FROM sales
    WHERE amount > (SELECT AVG(amount) FROM sales)
    GROUP BY product_name;

## Step 5: Cleanup

In [ ]:
DROP DATABASE IF EXISTS DEMO_DB;
DROP WAREHOUSE IF EXISTS DEMO_WH;
USE WAREHOUSE COMPUTE_WH;